# NLI base results: mDeBERTa-v3-base-mnli-xnli on SNLI & TrGLUE

Loads [yilmazzey/sdp2-nli](https://huggingface.co/datasets/yilmazzey/sdp2-nli) (snli_tr_1_1, trglue_mnli) and runs **test-only** evaluation with this model.

**No prompts:** BERT NLI is sequence-pair classification (premise [SEP] hypothesis → label).

**Splits:** snli → `test`; trglue → `test_matched`/`test_mismatched`.

**Metrics:** Accuracy, macro F1, per-class F1, confusion matrix (CSV + plot). Model is multilingual fine-tuned for NLI (~184M params, supports Turkish via XNLI; pretrained 3-class head, ~80% expected on similar data).

In [2]:
REPO_ID = "yilmazzey/sdp2-nli"
CONFIGS = ["snli_tr_1_1", "trglue_mnli"]
MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
NUM_LABELS = 3  # entailment, neutral, contradiction
RESULTS_DIR = "results_snli_trglue"
BATCH_SIZE = 32
EVAL_SPLITS = {
    "snli_tr_1_1": ["test"],
    "trglue_mnli": ["test_matched", "test_mismatched"],
}

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False

LABEL_NAMES = ["entailment", "neutral", "contradiction"]

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Load all dataset configs
datasets = {}
for cfg in CONFIGS:
    print(f"Loading {REPO_ID} :: {cfg} ...")
    datasets[cfg] = load_dataset(REPO_ID, cfg)
    print("  splits:", list(datasets[cfg].keys()))

Loading yilmazzey/sdp2-nli :: snli_tr_1_1 ...


  splits: ['train', 'validation', 'test']
Loading yilmazzey/sdp2-nli :: trglue_mnli ...
  splits: ['train', 'validation_matched', 'validation_mismatched', 'test_matched', 'test_mismatched']


In [ ]:
# Concatenate all test splits into one combined dataset
test_datasets = []

# SNLI test
if "test" in datasets["snli_tr_1_1"]:
    test_datasets.append(datasets["snli_tr_1_1"]["test"])
    print(f"SNLI test: {len(datasets['snli_tr_1_1']['test'])} examples")

# TrGLUE test_matched
if "test_matched" in datasets["trglue_mnli"]:
    test_datasets.append(datasets["trglue_mnli"]["test_matched"])
    print(f"TrGLUE test_matched: {len(datasets['trglue_mnli']['test_matched'])} examples")

# TrGLUE test_mismatched
if "test_mismatched" in datasets["trglue_mnli"]:
    test_datasets.append(datasets["trglue_mnli"]["test_mismatched"])
    print(f"TrGLUE test_mismatched: {len(datasets['trglue_mnli']['test_mismatched'])} examples")

# Concatenate all test datasets
combined_test = concatenate_datasets(test_datasets)
print(f"\nCombined test dataset: {len(combined_test)} examples")
print(f"Label distribution: {dict(Counter(combined_test['label']))}")

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, ignore_mismatched_sizes=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print(f"Using device: {device}")
print("Model loaded.")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 2482.79it/s, Materializing param=pooler.dense.weight]                                       
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cpu
Model loaded.


In [6]:
def tokenize_fn(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        max_length=256,
    )


def run_inference(ds):
    remove_cols = [c for c in ds.column_names if c != "label"]
    ds = ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=remove_cols,
        desc="Tokenize",
    )
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def collate_fn(examples):
        labels = torch.tensor([ex["label"] for ex in examples])
        batch = data_collator([{k: v for k, v in ex.items() if k != "label"} for ex in examples])
        batch["labels"] = labels
        return batch

    # Lower BATCH_SIZE to 16/8 if slow
    loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, collate_fn=collate_fn)
    preds_list, labels_list = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            out = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
            preds_list.append(out.logits.argmax(-1).cpu().numpy())
            labels_list.append(batch["labels"].numpy())
    y_pred = np.concatenate(preds_list)
    y_true = np.concatenate(labels_list)
    return y_true, y_pred

In [7]:
def compute_metrics(y_true, y_pred):
    acc = float(accuracy_score(y_true, y_pred))
    f1_macro = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    f1_per_class = {LABEL_NAMES[i]: float(f1_per_class[i]) for i in range(NUM_LABELS)}
    cm = confusion_matrix(y_true, y_pred)
    out = {"accuracy": acc, "f1_macro": f1_macro, "f1_per_class": f1_per_class}
    return out, cm


def save_confusion_plot(cm, path):
    if not HAS_PLOT:
        return
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

In [ ]:
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
all_metrics = {}

# Evaluate on combined test dataset
print("Evaluating on combined SNLI + TrGLUE test set...")
y_true, y_pred = run_inference(combined_test)
print("  True label dist:", dict(Counter(y_true)))
print("  Pred label dist:", dict(Counter(y_pred)))
metrics, cm = compute_metrics(y_true, y_pred)
all_metrics["combined_snli_trglue"] = {"test": metrics}

cm_path = Path(RESULTS_DIR) / "confusion_combined_test.csv"
np.savetxt(cm_path, cm, fmt="%d", delimiter=",")
save_confusion_plot(cm, Path(RESULTS_DIR) / "confusion_combined_test.png")

print(f"  accuracy={metrics['accuracy']:.4f}, f1_macro={metrics['f1_macro']:.4f}")

with open(Path(RESULTS_DIR) / "metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved {RESULTS_DIR}/metrics.json")

Evaluating snli_tr_1_1 / test ...


Inference:  42%|████▏     | 128/307 [03:30<04:54,  1.65s/it]


KeyboardInterrupt: 

In [ ]:
# Summary: Combined SNLI + TrGLUE results
for dataset_name, splits in all_metrics.items():
    for split_name, m in splits.items():
        print(f"{dataset_name} / {split_name}: acc={m['accuracy']:.4f}, F1_macro={m['f1_macro']:.4f}, F1_per_class={m['f1_per_class']}")